# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`@mbse/expressions/Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the term to bind the holes, then
instantiates the other side, a writer of the target dialect's term that the caller writes in a transaction. A dialect
with no binding gets a let's value substituted for its name, and imports are dropped where the other side cannot
declare them, or added where the translation needs them (`import numpy as np`). Translations keep an expression's
form; its value follows each dialect's rules, and this suite shows where those agree and where they do not.
(Evaluating the NumPy style with numpy is the Python implementation's only.)

In [ ]:
import { Expressions as E, Framework } from "@mbse/expressions";
import * as Patterns from "@mbse/expressions/Framework/Patterns";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as CppDialect from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import { Terms as F } from "@mbse/expressions/Framework";
import { Convert, Elementwise, Elide, Hole, Inline, Pairwise, Pattern, Prelude, Rule, holes, renames } from "@mbse/expressions/Framework/Translators";
import * as Translators from "@mbse/expressions/Translators";
import { Builders, Errors, Repr } from "@mbse/schemas/Framework";
import { assert, form_equal, raises, same } from "./support.js";

const { AttributeError, KeyError, LookupError, ValueError } = Errors;
const V = Patterns.Evaluators;

const store: any = new Builders.Store(); // every dialect's kinds, and the data the expressions are evaluated over
for (const module of [Framework, Patterns, Python3, Matlab, Excel, Latex, CDialect, CppDialect, SystemVerilog]) {
  store.transaction(module.install);
}

function data(txn: any): any {
  const S = txn.Schemas;
  const ints = () => S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt]));
  return txn.uid("trn.data/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Bus: S.OfObject })
    .spec([
      (_: any) => _.Person.name("Person").ref(true).properties({ name: S.OfStr, age: S.OfInt, email: S.OfStr }),
      (_: any) => _.Bus.name("Bus").ref(true).properties({ pins: ints(), allowed: ints() }),
    ])]))]);
}

store.transaction(data);

/** The term value `spec` (a writer of any dialect, a native value as a Patterns literal, or a term) writes. */
function term(spec: unknown): any {
  return store.transaction(() => store.Expressions.Framework.Term(E.DIALECT.spec(spec)));
}

/** The attributes of the term `spec` writes: a literal's `value`, say. */
function attributes(spec: unknown): ReadonlyMap<string, any> {
  return F.form(term(spec)).attributes;
}

const [P, M, X, L] = [Python3.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions];
const [PV, MV, XV] = [Python3.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [C, CV] = [CDialect.Expressions, CDialect.Evaluators];
const [SV, SVV] = [SystemVerilog.Expressions, SystemVerilog.Evaluators];
const [PATTERNS, PYTHON, MATLAB, EXCEL, LATEX] = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT];
const DIALECTS = [PATTERNS, PYTHON, MATLAB, EXCEL, LATEX];
const EVALUATE = new Map<F.Dialect, (e: unknown, scope?: any) => unknown>([
  [PATTERNS, V.OfAny as any], [PYTHON, PV.OfAny], [MATLAB, MV.OfAny], [EXCEL, XV.OfAny]]);
const NUMPY = Translators.Patterns_Python3.NUMPY;
const person = E.variable("person");
const age = person.age;
const core = term(age.ge(18n).and_(age.lt(65n).or_(person.has("email").not_())).and_(person.get("name").ne("x")).or_(
  age.add(1n).sub(2n).mul(3n).neg().gt(0n).and_(age.eq(1n).not_())).and_(person.has("name").eq(true)).and_(
  E.literal(1.5).le(2.5))); // every translatable operation and literal type, operands of one type, no let
const constraint = term(E.let_("a", age, E.variable("a").ge(18n).implies(person.has("email")))); // a let and implies
const permutations = (items: F.Dialect[], n: number): F.Dialect[][] =>
  n === 0 ? [[]] : items.flatMap((item) => permutations(items.filter((i) => i !== item), n - 1).map((rest) => [item, ...rest]));
const json = (value: unknown) => globalThis.JSON.stringify(value);
/** `expression`, a term of `source`, translated to `target` and written: itself when they are the same. */
const into = (source: F.Dialect, target: F.Dialect, expression: any): any =>
  (source === target ? expression : term(Translators.between(source, target).forward(expression)));
const people: any[] = store.transaction(() => [store.Person((p: any) => p.name("Ann").age(30n).email("a@x")),
  store.Person((p: any) => p.name("Bob").age(70n)), store.Person((p: any) => p.name("x").age(10n).email("c@x"))]);
const [PText, MText, XText, LText, CText, SVText] = [Python3.Text, Matlab.Text, Excel.Text, Latex.Text, CDialect.Text, SystemVerilog.Text];

<a id="TRN-01"></a>
## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
{
  const [A, B_] = holes("A", "B") as [Hole, Hole];
  const K = new Hole("K", String);
  const double = new Pattern("call", { callable: "add" }, A, A);
  assert(double.size() === 2 && String(double) === "call(A, A, callable='add')" && json([...double.holes()]) === json([["A", "argument"]]));
  const has = new Pattern("call", { callable: "has" }, A, new Pattern("literal", { value: K }));
  assert(has.size() === 3 && json([...has.holes()]) === json([["A", "argument"], ["K", "attribute"]]));
  raises(ValueError, () => new Rule(new Pattern("call", { callable: "f" }, A), new Pattern("call", { callable: "f" }, A, B_)),
    "call(A, B, callable='f') needs the argument hole B, which call(A, callable='f') does not bind");
  raises(ValueError, () => new Rule(new Pattern("call", { callable: "f" }, A), new Pattern("variable", { name: A })),
    "variable(name=A) needs the attribute hole A");
  new Rule(new Pattern("call", { callable: "f" }, A), new Pattern("call", { callable: "f" }, A, B_), "backward"); // only the other way needs B
  raises(ValueError, () => new Rule(double, double, "up"), "direction must be 'both', 'forward' or 'backward', got 'up'");
  raises(ValueError, () => new Elementwise("list", "list", {}, {}, "up"), "direction must be 'both', 'forward' or 'backward', got 'up'");
  for (const declare of [() => new Inline("let", "up"), () => new Elide("import", "up"),
    () => new Prelude(new Pattern("import", {}, A), "up")]) {
    raises(ValueError, declare, "side must be 'left' or 'right', got 'up'");
  }
  for (const pattern of [new Pattern("import", { module: "numpy" }),
    new Pattern("import", {}, new Pattern("name", { name: "x" })), new Pattern("import", { module: new Hole("M") }, A)]) {
    raises(ValueError, () => new Prelude(pattern, "right"), "a prelude is an import whose one argument is a hole");
  }
  assert(json(renames("call", "callable", "compare", "operator", { eq: "==" }, 2).map((r) =>
    [Object.fromEntries(r.left.attributes), Object.fromEntries(r.right.attributes)])) === json([[{ callable: "eq" }, { operator: "==" }]]));
  // A conversion translates literals by functions of their attributes, which give null where there is no counterpart.
  const doubleInts = new Convert("literal", "literal", (a) => (typeof a.value === "bigint" ? { value: a.value * 2n } : null),
    (a) => ({ value: (a.value as bigint) / 2n }));
  const doubling = new Pairwise(PATTERNS, PATTERNS, [doubleInts]);
  assert(attributes(doubling.forward(term(3n))).get("value") === 6n && attributes(doubling.backward(term(6n))).get("value") === 3n);
  assert(attributes(doubling.inverse().forward(term(6n))).get("value") === 3n);
  assert(attributes(doubling.inverse().backward(term(3n))).get("value") === 6n);
  raises(ValueError, () => doubling.forward(term("a")), "Patterns literal 'a' has no Patterns counterpart");
  for (const [left, right, named_] of [["call", "literal", "Patterns 'call'"], ["literal", "nothing", "Patterns 'nothing'"]]) {
    raises(ValueError, () => new Pairwise(PATTERNS, PATTERNS, [new Convert(left as string, right as string, doubleInts.forward, doubleInts.backward)]),
      `a conversion translates literals, and ${named_} is not one`);
  }

  // A translator within one dialect is a rewriting: holes bound twice must bind equal terms, or equal values.
  const simplify = new Pairwise(PATTERNS, PATTERNS, [
    new Rule(double, new Pattern("call", { callable: "mul" }, A, new Pattern("literal", { value: 2n })), "forward"),
    new Rule(new Pattern("call", { callable: "eq" }, new Pattern("literal", { value: K }), new Pattern("literal", { value: K })),
      new Pattern("literal", { value: true }), "forward"),
    new Rule(new Pattern("call", { callable: "add" }, A, B_), new Pattern("call", { callable: "add" }, A, B_), "forward"),
    new Rule(new Pattern("call", { callable: "eq" }, new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("W") })),
      new Pattern("literal", { value: false }), "forward"),
    new Rule(new Pattern("variable", { name: new Hole("N") }), new Pattern("variable", { name: new Hole("N") })),
    new Rule(new Pattern("literal", { value: new Hole("V") }), new Pattern("literal", { value: new Hole("V") }), "forward"),
  ]);
  const [x, y] = [E.variable("x"), E.variable("y")];
  assert(F.same(term(simplify.forward(term(x.add(x)))), term(x.mul(2n))));
  assert(F.same(term(simplify.forward(term(x.add(y)))), term(x.add(y))));
  assert(attributes(simplify.forward(term(E.literal("a").eq("a")))).get("value") === true);
  assert(attributes(simplify.forward(term(E.literal("a").eq("b")))).get("value") === false);
  assert(attributes(simplify.forward(term(E.literal(1n).eq(1n)))).get("value") === false); // K holds only a str
  assert(String(simplify) === "<translator Patterns <-> Patterns>");
  // A prelude of an import that binds no names never wraps a translation.
  const plain = new Pairwise(PATTERNS, MATLAB, [new Rule(new Pattern("variable", { name: new Hole("N") }), new Pattern("variable", { name: new Hole("N") })),
    new Prelude(new Pattern("import", { name: "geo.*" }, A), "right")]);
  assert(F.same(term(plain.forward(term(x))), term(M.identifier("x")))); // a MATLAB term
  // A translation is a writer: written where its caller wants it, as an expression's root.
  const doubled = simplify.forward(term(x.add(x)));
  const expression = store.transaction(() => store.Expressions.Framework.Expression((e: any) => e.root(doubled)));
  assert(F.same(F.term(expression), term(x.mul(2n))) && simplify.forward(term(x)) instanceof F.Writer);
}

<a id="TRN-02"></a>
## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
{
  assert(Translators.TRANSLATORS.length === 13); // every pair of the five, and Patterns with C, Cpp and SystemVerilog
  for (const [source, target] of permutations(DIALECTS, 2) as [F.Dialect, F.Dialect][]) {
    const translator = Translators.between(source, target);
    assert(translator.left() === source && translator.right() === target);
    const start = into(PATTERNS, source, core);
    const there = term(translator.forward(start));
    assert(target.validate(there, ["person"], true).length === 0, `${source.name()} ${target.name()}`);
    assert(F.same(term(translator.backward(there)), start), `${source.name()} ${target.name()}`); // round trip
    assert(F.same(term(translator.inverse().backward(start)), there));
    // a translation writes the target dialect's holder: its root narrowed to the target's Term, validated as the target
    const held = store.transaction(() => target.path.split(".").reduce((at: any, part) => at[part], store.Expressions).Expression((e: any) => e.root(translator.forward(start))));
    assert(held.as_schema() === store.registered(`Expressions.${target.path}.Expression`) && F.same(F.term(held), there));
    assert(target.validate(held, ["person"], true).length === 0);
    assert(F.same(term(translator.backward(held)), start)); // and a translator reads a holder as any expression
  }
  raises(LookupError, () => Translators.between(PATTERNS, PATTERNS), "no translator between Patterns and Patterns");

  assert(PText.ToText(into(PATTERNS, PYTHON, constraint)) === "(lambda a: hasattr(person, 'email') if a >= 18 else True)(person.age)");
  assert(MText.ToText(into(PATTERNS, MATLAB, constraint)) === '~(person.age >= 18) || isfield(person, "email")');
  assert(XText.ToText(into(PATTERNS, EXCEL, constraint)) === "=LET(a, person.age, IF(a >= 18, NOT(ISERROR(person.email)), TRUE))");
  assert(LText.ToText(into(PATTERNS, LATEX, constraint))
    === "a \\geq 18 \\implies \\operatorname{has}(\\mathit{person}, \\text{email}) \\quad \\text{where } a = \\mathit{person}.\\mathit{age}");
  assert(LText.ToText(into(PYTHON, LATEX, term(P.binop("/", P.name("x"), 2n)))) === "\\frac{x}{2}");
  // Python, Excel and LaTeX have lets and implication, so the constraint round-trips; MATLAB inlines the let and writes
  // implies with ||.
  for (const other of [PYTHON, EXCEL, LATEX]) assert(F.same(into(other, PATTERNS, into(PATTERNS, other, constraint)), constraint));
  const back = into(MATLAB, PATTERNS, into(PATTERNS, MATLAB, constraint));
  assert(F.same(back, term(age.ge(18n).not_().or_(person.has("email")))));

  // NumPy style is a second translator between Patterns and Python: numpy's functions, over columns, importing numpy.
  const styled = term(NUMPY.forward(constraint));
  assert(PText.ToText(styled) === "import numpy as np\n(lambda a: np.where(np.greater_equal(a, 18), "
    + "np.logical_not(np.ma.getmaskarray(person['email'])), True))(person['age'])");
  assert(F.same(term(NUMPY.backward(styled)), constraint) && F.same(term(NUMPY.backward(term(NUMPY.forward(core)))), core));
  assert(PText.ToText(term(NUMPY.forward(term(E.literal(1n).add(2n))))) === "import numpy as np\nnp.add(1, 2)");
  assert(PText.ToText(term(NUMPY.forward(term(1n)))) === "1" && PText.ToText(term(NUMPY.forward(term(person)))) === "person"); // no np, no import
  assert(F.same(term(NUMPY.inverse().forward(styled)), constraint));
  assert(attributes(NUMPY.inverse().backward(term(1n))).get("value") === 1n);
  const imported = term(P.import_("math", P.importfrom("numpy", "pi", P.compare(">=", P.attribute(P.name("person"), "age"), 18n))));
  // imports the other side cannot declare are dropped
  for (const target of [PATTERNS, MATLAB, EXCEL]) assert(F.same(into(PYTHON, target, imported), into(PATTERNS, target, term(age.ge(18n)))));
  assert(XText.ToText(into(MATLAB, EXCEL, term(M.import_("pkg.*", M.binary(">", 1n, 0n))))) === "=1 > 0");
  assert(F.same(into(MATLAB, PATTERNS, term(M.import_("pkg.*", M.constant(1n)))), term(1n)));
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(term(P.DIALECT.spec((t: any) => t.import((i: any) => i.module("math"))))),
    "an import needs a body");
  raises(ValueError, () => Translators.between(PYTHON, PATTERNS).forward(term(P.import_("numpy", P.call("np.add", 1n, 2n), "np"))),
    "Python3 call has no Patterns counterpart"); // read by NUMPY
}

<a id="TRN-03"></a>
## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for (const [a, b, c] of permutations(DIALECTS, 3) as [F.Dialect, F.Dialect, F.Dialect][]) {
  const start = into(PATTERNS, a, core);
  const direct = into(a, c, start);
  const routed = into(b, c, into(a, b, start));
  assert(F.same(direct, routed), `${a.name()} ${b.name()} ${c.name()}`);
}

<a id="TRN-04"></a>
## TRN-04 · Co-traversal translates each term once, keeps sharing, and traces the correspondence

In [ ]:
{
  // A term is a nested value, so what was shared is written at each use: each use is translated once, and the trace
  // pairs each source term with the writer of its translation, a term after the terms within it.
  const shared = term(E.operation("add", age, age));
  let trace: [any, any][] = [];
  const out = term(Translators.between(PATTERNS, EXCEL).forward(shared, trace));
  const [left, right] = F.form(out).arguments;
  assert(F.same(left, right) && XText.ToText(out) === "=person.age + person.age");
  assert(json(trace.map(([s, t]) => [F.kindOf(s)[0], F.kindOf(term(t))[0]])) === json([["variable", "variable"],
    ["call", "field"], ["variable", "variable"], ["call", "field"], ["call", "infix"]]));
  assert(F.same(trace[4]?.[0], shared) && F.same(term(trace[4]?.[1]), out) && F.same(term(trace[1]?.[1]), left));
  const a = E.variable("a");
  trace = [];
  const inlined = term(Translators.between(PATTERNS, MATLAB).forward(term(E.let_("a", age, a.add(a))), trace));
  const [first, second] = F.form(inlined).arguments;
  assert(MText.ToText(inlined) === "person.age + person.age" && F.same(first, second));
  assert(same(trace.map(([s]) => F.kindOf(s)[0]), ["variable", "call", "variable", "variable", "call", "let"])); // the value once
  const shadowed = term(E.let_("a", 1n, E.variable("a").add(E.let_("a", 2n, E.variable("a"))))); // the innermost binding wins
  assert(MText.ToText(into(PATTERNS, MATLAB, shadowed)) === "1 + 2");
  assert(MV.OfAny(into(PATTERNS, MATLAB, shadowed)) === 3 && V.OfAny(shadowed) === 3n);
  assert(MText.ToText(into(EXCEL, MATLAB, term(X.let_("n", 2n, X.infix("*", X.name("n"), X.name("n")))))) === "2 .* 2");
  raises(TypeError, () => Translators.between(PATTERNS, MATLAB).forward(7n), "not a term: 7");
  raises(ValueError, () => Translators.between(PATTERNS, MATLAB).forward(term((t: any) => t.let((l: any) => l.name("a").body(E.variable("a"))))),
    "a let needs a value");
  raises(ValueError, () => Translators.between(EXCEL, MATLAB).forward(term(X.DIALECT.spec((t: any) => t.let((l: any) => l.name("a").value(X.constant(1n)))))),
    "a let needs a body");
}

<a id="TRN-05"></a>
## TRN-05 · What a dialect cannot say is refused, naming the term

In [ ]:
for (const [translate, expression, message] of [
  [(e: any) => Translators.between(PATTERNS, EXCEL).forward(e), E.literal(new Uint8Array([0])), "Patterns literal b'\\x00' has no Excel counterpart"],
  [(e: any) => Translators.between(PATTERNS, PYTHON).forward(e), (t: any) => t.literal((l: any) => l), "Patterns literal None has no Python3 counterpart"],
  [(e: any) => Translators.between(PATTERNS, MATLAB).forward(e), E.operation("entries", person, "ports"), "Patterns call 'entries' has no MATLAB counterpart"],
  [(e: any) => Translators.between(PATTERNS, PYTHON).forward(e), E.operation("get", person, E.variable("key")), "Patterns call 'get' has no Python3 counterpart"],
  [(e: any) => NUMPY.forward(e), E.operation("has", person, 7n), "Patterns call 'has'"],
  [(e: any) => Translators.between(PATTERNS, PYTHON).forward(e), E.operation("eq", 1n, 2n, 3n), "Patterns call 'eq' has no Python3 counterpart"],
  [(e: any) => Translators.between(EXCEL, PATTERNS).forward(e), X.function("ISERROR", 1n), "Excel call 'ISERROR' has no Patterns counterpart"],
  [(e: any) => Translators.between(EXCEL, PATTERNS).forward(e), X.function("IF", true, 1n, 2n), "Excel call 'IF' has no Patterns counterpart"],
  [(e: any) => Translators.between(EXCEL, PATTERNS).forward(e), X.DIALECT.spec((t: any) => t.let((l: any) => l.name("a").value(X.constant(1n)))),
    "Excel let has no Patterns counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.ifexp(true, 1n, 2n), "Python3 ifexp has no MATLAB counterpart"],
  [(e: any) => Translators.between(EXCEL, MATLAB).forward(e), X.cell("A1"), "Excel cell has no MATLAB counterpart"],
  [(e: any) => Translators.between(PATTERNS, LATEX).forward(e), E.literal(new Uint8Array([0])), "Patterns literal b'\\x00' has no LaTeX counterpart"],
  [(e: any) => Translators.between(LATEX, PATTERNS).forward(e), L.frac(1n, 2n), "LaTeX frac has no Patterns counterpart"],
  [(e: any) => Translators.between(PYTHON, LATEX).forward(e), P.ifexp(true, 1n, 2n), "Python3 ifexp has no LaTeX counterpart"],
  [(e: any) => Translators.between(MATLAB, PATTERNS).forward(e), M.call("isfield", M.identifier("s"), M.identifier("k")), "MATLAB call 'isfield'"],
  [(e: any) => Translators.between(PYTHON, PATTERNS).forward(e), P.DIALECT.spec((t: any) => t.call((c: any) => c.args([P.name("x"), P.constant("k")]))),
    "Python3 call has no Patterns counterpart"], // a call of no function: not hasattr's
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.constant(new Uint8Array([0])), "Python3 literal b'\\x00' has no MATLAB counterpart"],
  // Python3's kinds with no counterpart, and what the narrower kinds could not hold, are refused by name
  [(e: any) => Translators.between(PYTHON, PATTERNS).forward(e), P.lambda_(1n), "Python3 lambda has no Patterns counterpart"],
  [(e: any) => Translators.between(PYTHON, PATTERNS).forward(e), P.constant(null), "Python3 constant None of NoneType has no Patterns counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.compare("<", 1n, 2n, ["is", 3n]), "Python3 compare has no MATLAB counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.boolop("and", 1n), "Python3 boolop 'and' has no MATLAB counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.DIALECT.spec((t: any) => t.compare((r: any) => r.left(P.constant(1n)))),
    "Python3 compare has no MATLAB counterpart"],
  [(e: any) => Translators.between(PYTHON, MATLAB).forward(e), P.call("hasattr", P.name("s"), "k", P.keyword("k", 1n)), "Python3 call has no MATLAB counterpart"],
  [(e: any) => Translators.between(PYTHON, PATTERNS).forward(e), P.dict_([null, P.name("m")]), "Python3 dict has no Patterns counterpart"],
] as [(e: unknown) => unknown, unknown, string][]) {
  raises(ValueError, () => translate(term(expression)), message);
}
// A chain translates as the `and` of its links, its middle operands written in each, and `and` or `or` of more than two
// values as two, nested: each read as its kind unfolds it, where no rule reads it.
const chain = term(P.compare("<", 1n, P.name("x"), ["<=", 3n], ["<", 4n]));
assert(F.same(term(Translators.between(PYTHON, PATTERNS).forward(chain)), term(
  E.literal(1n).lt(E.variable("x")).and_(E.variable("x").le(3n)).and_(E.literal(3n).lt(4n)))));
assert(XText.ToText(term(Translators.between(PYTHON, EXCEL).forward(term(P.boolop("or", 1n, 2n, 3n))))) === "=OR(OR(1, 2), 3)");
const traced: [unknown, unknown][] = [];
Translators.between(PYTHON, MATLAB).forward(chain, traced); // the unfolded forms traced as the terms are
assert(traced.filter(([source]) => source instanceof F.Form).length === 6 && F.same(traced[traced.length - 1]?.[0], chain));
// A literal translates by its value: its spelling is its own dialect's, and one with no value has none to translate.
const spelled = term(P.DIALECT.spec((t: any) => t.literal((r: any) => r.value((v: any) => v.as_scalar(16n)).text("0x10").base(16n))));
assert(PText.ToText(spelled) === "0x10" && MText.ToText(into(PYTHON, MATLAB, spelled)) === "16");
assert(form_equal(F.form(into(PYTHON, PATTERNS, spelled)), new F.Form("literal", { value: 16n })));
raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(term(P.DIALECT.spec(
  (t: any) => t.literal((r: any) => r.text("08").problems(["bad"]))))), "Python3 literal None has no MATLAB counterpart");

<a id="TRN-06"></a>
## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
{
  const truthy = (value: unknown) => PV.truthy(value);
  const EVALUATED = DIALECTS.filter((d) => EVALUATE.has(d)); // LaTeX has no evaluator
  for (const someone of people) {
    for (const expression of [core, constraint]) {
      const values = EVALUATED.map((d) => truthy(EVALUATE.get(d)?.(into(PATTERNS, d, expression), { person: someone })));
      assert(new Set(values).size === 1, `${someone.name.get()} ${values}`);
    }
  }
  // NumPy evaluates all three at once, over columns: the Python implementation's only.

  const nobody = store.transaction(() => store.Person((p: any) => p.name("Nobody"))); // no age: each dialect's own notion of a missing value
  const missing = term(age.ge(18n));
  assert(V.OfAny(missing, { person: nobody }) === null); // Patterns: unknown
  raises(AttributeError, () => PV.OfAny(into(PATTERNS, PYTHON, missing), { person: nobody }), "'Person' object has no attribute 'age'"); // Python raises
  assert(XV.OfAny(into(PATTERNS, EXCEL, missing), { person: nobody }) === XV.Error.of("#FIELD!")); // an error value
  raises(KeyError, () => MV.OfAny(into(PATTERNS, MATLAB, missing), { person: nobody }), 'Unrecognized field name "age".'); // MATLAB raises
  const [mixed, cased] = [term(E.literal(1n).eq(1.0)), term(E.literal("a").eq("A"))];
  assert(V.OfAny(mixed) === null && V.OfAny(cased) === false); // Patterns never coerces, and compares text exactly
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(PATTERNS, d, mixed)))), [true, true, true]));
  assert(same(EVALUATED.slice(1).map((d) => truthy(EVALUATE.get(d)?.(into(PATTERNS, d, cased)))), [false, false, true]));
}

<a id="TRN-07"></a>
## TRN-07 · The bitwise operations translate between Patterns, Python, MATLAB and Excel, and evaluate alike on non-negative integers

In [ ]:
{
  const x = E.variable("x");
  const bits = term(x.bitand(12n).bitor(x.shr(2n)).bitxor(x.shl(1n)));
  const RENDER = new Map<F.Dialect, (e: any) => string>([[PYTHON, PText.ToText], [MATLAB, MText.ToText], [EXCEL, XText.ToText]]);
  const expected = new Map<F.Dialect, string>([[PYTHON, "(x & 12 | x >> 2) ^ x << 1"],
    [MATLAB, "bitxor(bitor(bitand(x, 12), bitshift(x, -2)), bitshift(x, 1))"],
    [EXCEL, "=BITXOR(BITOR(BITAND(x, 12), BITRSHIFT(x, 2)), BITLSHIFT(x, 1))"]]);
  for (const [dialect, source] of expected) {
    const translator = Translators.between(PATTERNS, dialect);
    const translated = term(translator.forward(bits));
    assert(RENDER.get(dialect)!(translated) === source && F.same(term(translator.inverse().forward(translated)), bits));
    assert(Number(EVALUATE.get(dialect)!(translated, { x: 13n })) === 21 && V.OfAny(bits, { x: 13n }) === 21n);
  }
  // Between the other dialects too, by any route.
  for (const [a, b] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Dialect, F.Dialect][]) {
    const source = into(PATTERNS, a, bits);
    const target = into(a, b, source);
    assert(F.same(target, into(PATTERNS, b, bits)) && F.same(into(b, a, target), source));
  }
  // bitnot is Python's ~ and NumPy's invert; MATLAB and Excel have none for their numbers.
  assert(PText.ToText(into(PATTERNS, PYTHON, term(x.bitnot()))) === "~x");
  assert(PText.ToText(term(NUMPY.forward(term(x.bitnot().shl(1n))))) === "import numpy as np\nnp.left_shift(np.invert(x), 1)");
  raises(ValueError, () => Translators.between(PATTERNS, MATLAB).forward(term(x.bitnot())), "Patterns call 'bitnot' has no MATLAB counterpart");
  raises(ValueError, () => Translators.between(PATTERNS, EXCEL).forward(term(x.bitnot())), "Patterns call 'bitnot' has no Excel counterpart");
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(term(P.unaryop("~", P.name("x")))),
    "Python3 unaryop '~' has no MATLAB counterpart");
  // Patterns' and Python3's integers are unbounded and signed; MATLAB's and Excel's bit functions are not.
  const mask = term(x.bitand(1n));
  assert(V.OfAny(mask, { x: -1n }) === 1n && PV.OfAny(into(PATTERNS, PYTHON, mask), { x: -1n }) === 1n);
  assert(XV.OfAny(into(PATTERNS, EXCEL, mask), { x: -1n }) === XV.Error.of("#NUM!"));
}

<a id="TRN-08"></a>
## TRN-08 · Patterns and C, and Patterns and C++, translate into each other by one rule set; lets are inlined, and `has`, `->`, typed literals, casts and the other kinds have no counterpart

In [ ]:
{
  const [CCPP, CPP] = [C.DIALECT, CppDialect.Expressions.DIALECT];
  const [x, person] = [E.variable("x"), E.variable("person")];
  const constraint = term(E.let_("a", person.age, E.variable("a").ge(18n).and_(person.width.shl(2n).bitand(255n).ne(0n)).implies(person.get("name").eq("x"))));
  const translator = Translators.between(PATTERNS, CCPP);
  const translated = term(translator.forward(constraint));
  assert(CText.ToText(translated) === '!(person.age >= 18 && (person.width << 2 & 255) != 0) || person.name == "x"');
  const expected = term(E.operation("or", E.operation("not", person.age.ge(18n).and_(person.width.shl(2n).bitand(255n).ne(0n))), person.get("name").eq("x")));
  assert(F.same(term(translator.inverse().forward(translated)), expected));
  const every = term(x.add(1n).sub(2n).mul(3n).neg().bitor(x.bitxor(1n)).bitand(x.bitnot()).shr(1n).lt(x).or_(x.le(1n).and_(x.gt(2n).not_())).ne(
    x.ge(0n).eq(E.literal(true))));
  assert(F.same(into(CCPP, PATTERNS, into(PATTERNS, CCPP, every)), every) && CText.ToText(into(PATTERNS, CCPP, term(1.5))) === "1.5");
  const check = term(x.add(1n).mul(2n).gt(5n).and_(x.lt(9n)));
  for (const value of [3n, 9n]) { // C's ints and Patterns' agree where neither overflows
    assert(CV.OfAny(into(PATTERNS, CCPP, check), { x: value }) === V.OfAny(check, { x: value }));
  }
  // A literal translates by its value, its spelling (its text, quote, base and digit separators) aside; one of a type of
  // its own (a suffix or a prefix), or a cast, has no Patterns counterpart yet: a term carries no domain, and its
  // conversions wait for domain inference.
  const hexadecimal = C.DIALECT.spec((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(255n)).text("0xFF").base(16n)));
  const character = C.DIALECT.spec((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'"))); // 'a', its code
  for (const [literal, plain] of [[C.literal(true), true], [C.literal(1.5), 1.5], [C.literal(5n), 5n], [hexadecimal, 255n], [character, 97n]] as const) {
    assert(F.same(into(CCPP, PATTERNS, term(literal)), term(plain)));
  }
  const refused: [unknown, F.Dialect, string][] = [
    [person.has("email"), PATTERNS, "Patterns call 'has' has no C counterpart"],
    [E.literal(new Uint8Array([1])), PATTERNS, "Patterns literal b'\\x01' has no C counterpart"],
    [E.list_(1n), PATTERNS, "Patterns list has no C counterpart"],
    [C.literal(5n, "u"), CCPP, "C literal 5 of unsigned int has no Patterns counterpart"],
    [C.literal(1.5, "f"), CCPP, "C literal 1.5 of float has no Patterns counterpart"],
    [C.DIALECT.spec((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'").prefix("L"))), CCPP,
      "C literal 97 of wchar_t has no Patterns counterpart"],
    [C.cast((t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("int"))])), 5n), CCPP,
      "C castexpression has no Patterns counterpart"],
    [C.member(C.identifier("x"), "a", "->"), CCPP, "C memberexpression has no Patterns counterpart"],
    [C.unary("*", C.identifier("x")), CCPP, "C unaryexpression '*' has no Patterns counterpart"],
    [C.DIALECT.spec((u: any) => u.sizeofexpression((s: any) => s.operand(C.identifier("x")))), CCPP, "C sizeofexpression has no Patterns counterpart"],
  ];
  for (const [expression, source, message] of refused) {
    raises(ValueError, () => Translators.between(source, source === CCPP ? PATTERNS : CCPP).forward(term(expression)), message);
  }
  // C++: the same rules (Patterns_C.RULES), over the kinds it shares with C, writing C++'s terms, a member's name an
  // identifier in both, but for a name used as an expression, C's variable and C++'s idexpression holding its identifier;
  // and C++'s string, whose quote is not its spelling, as `'a'` is a char in C++
  const shared = Translators.Patterns_C.RULES.filter((rule) => rule !== Translators.Patterns_C.IDENTIFIER);
  const held = Translators.Patterns_Cpp.RULES.slice(0, Translators.Patterns_C.RULES.length).filter((rule) => shared.includes(rule));
  assert(held.length === shared.length && shared.every((rule, i) => held[i] === rule)); // one rule set
  assert(Repr.repr(F.form(into(PATTERNS, CCPP, term(person.name))).attributes.get("member")) === "Part(kind='identifier', attributes={'spelling': 'name'}, size=0)");
  assert(F.form(into(PATTERNS, CCPP, term(x))).kind === "variable" && Repr.repr(F.form(into(PATTERNS, CCPP, term(x))).attributes) === "{'name': 'x'}");
  assert(F.form(into(PATTERNS, CPP, term(x))).kind === "idexpression"
    && Repr.repr(F.form(into(PATTERNS, CPP, term(x))).attributes) === "{'name': Part(kind='identifier', attributes={'spelling': 'x'}, size=0)}");
  assert(F.same(into(CPP, PATTERNS, term(CppDialect.Expressions.identifier("x"))), term(x)));
  const cout = CPP.spec((u: any) => u.idexpression((r: any) => r.name((n: any) => n.qualifiedname((q: any) => q.qualifiers([ // std::cout
    (m: any) => m.identifier((i: any) => i.spelling("std"))]).name((m: any) => m.identifier((i: any) => i.spelling("cout")))))));
  raises(ValueError, () => Translators.between(CPP, PATTERNS).forward(term(cout)), "Cpp idexpression has no Patterns counterpart"); // a name that is not an identifier
  const inCpp = into(PATTERNS, CPP, constraint);
  assert(F.dialectOf(inCpp) === CPP.union && CppDialect.Text.ToText(inCpp) === CText.ToText(translated));
  assert(F.same(into(CPP, PATTERNS, inCpp), expected) && F.same(into(CPP, PATTERNS, into(PATTERNS, CPP, every)), every));
  assert(CppDialect.Evaluators.OfAny(into(PATTERNS, CPP, check), { x: 3n }) === V.OfAny(check, { x: 3n }));
  raises(ValueError, () => Translators.between(CPP, PATTERNS).forward(term(CppDialect.Expressions.literal(5n, "u"))),
    "Cpp literal 5 of unsigned int has no Patterns counterpart");
  const quoted = CPP.spec((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar("s")).quote('"'))); // "s", written with its quote
  assert(F.same(into(CPP, PATTERNS, term(quoted)), term("s")) && F.same(into(PATTERNS, CPP, term("s")), term(CppDialect.Expressions.literal("s"))));
  raises(ValueError, () => Translators.between(CPP, PATTERNS).forward(term(CPP.spec((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(97n)).quote("'"))))),
    "Cpp literal 97 of char has no Patterns counterpart"); // 'a', a char: what the literal is, not kept silently as an int
  raises(ValueError, () => Translators.between(CPP, PATTERNS).forward(term(CPP.spec((u: any) => u.lambdaexpression((r: any) => r)))),
    "Cpp lambdaexpression has no Patterns counterpart"); // a kind of C++'s alone
  raises(LookupError, () => Translators.between(CCPP, CPP), "no translator between C and Cpp"); // C++ is not a superset of C
}

<a id="TRN-09"></a>
## TRN-09 · Patterns and SystemVerilog translate into each other; lets are inlined, bools are single bits, and `has`, sized vectors and 4-state comparisons have no counterpart

In [ ]:
{
  const SYSTEMVERILOG = SV.DIALECT;
  const [x, person] = [E.variable("x"), E.variable("person")];
  const constraint = term(E.let_("a", person.age, E.variable("a").ge(18n).and_(person.width.shr(2n).bitand(255n).ne(0n)).implies(person.get("name").eq("x"))));
  const translator = Translators.between(PATTERNS, SYSTEMVERILOG);
  const translated = term(translator.forward(constraint));
  assert(SVText.ToText(translated) === 'person.age >= 18 && (person.width >>> 2 & 255) != 0 -> person.name == "x"');
  const expected = term(person.age.ge(18n).and_(person.width.shr(2n).bitand(255n).ne(0n)).implies(person.get("name").eq("x")));
  assert(F.same(term(translator.inverse().forward(translated)), expected));
  const every = term(x.add(1n).sub(2n).mul(3n).neg().bitor(x.bitxor(1n)).bitand(x.bitnot()).shl(1n).lt(x).or_(x.le(1n).and_(x.gt(2n).not_())).ne(
    x.ge(0n).eq(E.literal(true)).or_(E.literal(false))));
  assert(F.same(into(SYSTEMVERILOG, PATTERNS, into(PATTERNS, SYSTEMVERILOG, every)), every));
  assert(SVText.ToText(into(PATTERNS, SYSTEMVERILOG, term(E.literal(true).and_(E.literal(1.5).lt(2n))))) === "1'b1 && 1.5 < 2");
  const check = term(x.add(1n).mul(2n).gt(5n).and_(x.lt(9n)));
  for (const value of [3n, 9n]) { // SystemVerilog's integers and Patterns' agree where neither overflows
    assert((SVV.OfAny(into(PATTERNS, SYSTEMVERILOG, check), { x: value }) as any).integer() === (V.OfAny(check, { x: value }) ? 1n : 0n));
  }
  // A sized vector has no Patterns counterpart yet but a bool's single bit: its conversions wait for domain inference.
  assert(F.same(into(SYSTEMVERILOG, PATTERNS, term(SV.vector("1"))), term(true)) && F.same(into(SYSTEMVERILOG, PATTERNS, term(SV.vector("0"))), term(false)));
  const refused: [unknown, F.Dialect, string][] = [
    [person.has("email"), PATTERNS, "Patterns call 'has' has no SystemVerilog counterpart"],
    [E.mapping(["a", 1n]), PATTERNS, "Patterns mapping has no SystemVerilog counterpart"],
    [SV.vector("11001000", null, "d"), SYSTEMVERILOG, "SystemVerilog vector '11001000' of logic [7:0] has no Patterns counterpart"],
    [SV.vector("10xz"), SYSTEMVERILOG, "SystemVerilog vector '10xz' of logic [3:0] has no Patterns counterpart"],
    [SV.binary("===", SV.identifier("x"), 1n), SYSTEMVERILOG, "SystemVerilog binary '===' has no Patterns counterpart"],
  ];
  for (const [expression, source, message] of refused) {
    raises(ValueError, () => Translators.between(source, source === SYSTEMVERILOG ? PATTERNS : SYSTEMVERILOG).forward(term(expression)), message);
  }
}

<a id="TRN-10"></a>
## TRN-10 · Collections translate to Python's builtins and generator expressions, and evaluate alike over lists

In [ ]:
const [bus, p] = [E.variable("bus"), E.variable("p")];
const collections = term(bus.pins.all("p", p.gt(0n)).and_(bus.pins.any("p", p.in_(bus.allowed))).and_(
  bus.pins.count_where("p", p.ge(2n)).le(bus.pins.count())).and_(
  bus.pins.item(0n).add(bus.pins.sum()).lt(bus.pins.max().mul(bus.pins.min()).add(100n))));
const unique = term(bus.pins.unique()); // written as an idiom, which does not read back
const buses: any[] = store.transaction(() => ([[[1n, 2n, 3n], [3n]], [[1n, 1n], [5n]], [[0n, 2n], [2n]]] as bigint[][][]).map(([pins, allowed]) =>
  store.Bus((b: any) => b.pins(pins).allowed(allowed))));
{
  const PY = Translators.between(PATTERNS, PYTHON);
  const there = term(PY.forward(collections));
  assert(PText.ToText(there) === "((all(p > 0 for p in bus.pins) and any(p in bus.allowed for p in bus.pins)) and "
    + "sum(1 for p in bus.pins if p >= 2) <= len(bus.pins)) and "
    + "bus.pins[0] + sum(bus.pins) < max(bus.pins) * min(bus.pins) + 100"); // Patterns' `and`s, nested
  assert(F.same(term(PY.backward(there)), collections) && F.same(term(PY.inverse().forward(there)), collections));
  assert(PText.ToText(term(PY.forward(unique))) === "len(set(bus.pins)) == len(bus.pins)");
  for (const b of buses) {
    for (const expression of [collections, unique]) {
      assert(V.OfAny(expression, { bus: b }) === PV.OfAny(term(PY.forward(expression)), { bus: b }), String(b.pins.get()));
    }
  }
  // The container constructors are Python's displays: a list display and a dict display, evaluating alike.
  const built = term(E.list_(bus.pins.count(), 2n).item(0n).add(E.mapping(["a", 1n], ["b", E.list_(3n)]).item("b").count()));
  const shown = term(PY.forward(built));
  assert(PText.ToText(shown) === "[len(bus.pins), 2][0] + len({'a': 1, 'b': [3]}['b'])" && F.same(term(PY.backward(shown)), built));
  assert(buses.every((b) => PV.OfAny(shown, { bus: b }) === V.OfAny(built, { bus: b })));
  const refused: [unknown, any, string][] = [
    [bus.entries("ports"), PY, "Patterns call 'entries' has no Python3 counterpart"],
    [bus.pins.count_where("p", p.gt(0n)), NUMPY, "Patterns quantifier 'count' has no Python3 counterpart"],
    [P.call("all", P.name("xs")), PY.inverse(), "Python3 call has no Patterns counterpart"],
    [P.call("sum", P.generatorexp(P.name("p"), P.comprehension("p", P.name("xs")))), PY.inverse(), "Python3 generatorexp has no Patterns counterpart"],
    [PY.forward(unique), PY.inverse(), "Python3 call has no Patterns counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(term(expression)), message);
}

<a id="TRN-11"></a>
## TRN-11 · Collections translate to SystemVerilog's array methods, and evaluate alike over arrays

In [ ]:
{
  const SVT = Translators.between(PATTERNS, SV.DIALECT);
  const there = term(SVT.forward(collections)); // TRN-10's
  assert(SVText.ToText(there) === "bus.pins.and(p) with (p > 0) && bus.pins.or(p) with (p inside {bus.allowed}) && "
    + "bus.pins.sum(p) with (int'(p >= 2)) <= bus.pins.size() && "
    + "bus.pins[0] + bus.pins.sum() < bus.pins.max()[0] * bus.pins.min()[0] + 100");
  assert(F.same(term(SVT.backward(there)), collections));
  assert(SVText.ToText(term(SVT.forward(unique))) === "bus.pins.unique().size() == bus.pins.size()"); // an idiom, which does not read back
  for (const b of buses) {
    for (const expression of [collections, unique]) {
      assert((SVV.OfAny(term(SVT.forward(expression)), { bus: b }) as any).integer() === (V.OfAny(expression, { bus: b }) ? 1n : 0n),
        String(b.pins.get()));
    }
  }
  const xs = SV.identifier("xs");
  const refused: [unknown, any, string][] = [
    [bus.entries("ports"), SVT, "Patterns call 'entries' has no SystemVerilog counterpart"],
    [SV.method(xs, "product"), SVT.inverse(), "SystemVerilog method 'product' has no Patterns counterpart"],
    [SV.iterate(xs, "xor", "p", SV.identifier("p")), SVT.inverse(), "SystemVerilog iterate 'xor' has no Patterns counterpart"],
    [SVT.forward(unique), SVT.inverse(), "SystemVerilog method 'unique' has no Patterns counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(term(expression)), message);
}

<a id="TRN-12"></a>
## TRN-12 · Collections translate to MATLAB's arrays: arrayfun, indexing from 1 and array functions, and evaluate alike

In [ ]:
{
  const MT = Translators.between(PATTERNS, MATLAB);
  const there = term(MT.forward(collections)); // TRN-10's
  assert(MText.ToText(there) === "all(arrayfun(@(p) p > 0, bus.pins)) && any(arrayfun(@(p) ismember(p, bus.allowed), bus.pins)) && "
    + "nnz(arrayfun(@(p) p >= 2, bus.pins)) <= numel(bus.pins) && "
    + "bus.pins(0 + 1) + sum(bus.pins) < max(bus.pins) .* min(bus.pins) + 100");
  assert(F.same(term(MT.backward(there)), collections));
  assert(MText.ToText(term(MT.forward(unique))) === "numel(unique(bus.pins)) == numel(bus.pins)"); // an idiom, which does not read back
  for (const b of buses) {
    for (const expression of [collections, unique]) {
      assert(MV.OfAny(term(MT.forward(expression)), { bus: b }) === V.OfAny(expression, { bus: b }), String(b.pins.get()));
    }
  }
  // A list is a concatenation, `[a, b]`, evaluating alike; a mapping has no counterpart.
  const listed = term(E.list_(bus.pins.count(), 2n).sum());
  assert(MText.ToText(term(MT.forward(listed))) === "sum([numel(bus.pins), 2])" && F.same(term(MT.backward(term(MT.forward(listed)))), listed));
  assert(buses.every((b) => Number(MV.OfAny(term(MT.forward(listed)), { bus: b })) === Number(V.OfAny(listed, { bus: b }))));
  raises(ValueError, () => MT.forward(term(E.mapping(["a", 1n]))), "Patterns mapping has no MATLAB counterpart");
  const xs = M.identifier("xs");
  const refused: [unknown, any, string][] = [
    [bus.entries("ports"), MT, "Patterns call 'entries' has no MATLAB counterpart"],
    [M.index(xs, 1n), MT.inverse(), "MATLAB index has no Patterns counterpart"], // only xs(i + 1) reads back
    [M.call("all", xs), MT.inverse(), "MATLAB call 'all' has no Patterns counterpart"],
    [MT.forward(unique), MT.inverse(), "MATLAB call 'unique' has no Patterns counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(term(expression)), message);
}

<a id="TRN-13"></a>
## TRN-13 · Collections translate to Excel's arrays: MAP and LAMBDA, INDEX and MATCH and array functions, and evaluate alike

In [ ]:
{
  const XT = Translators.between(PATTERNS, EXCEL);
  const there = term(XT.forward(collections)); // TRN-10's
  assert(XText.ToText(there) === "=AND(AND(AND(AND(MAP(bus.pins, LAMBDA(p, p > 0))), OR(MAP(bus.pins, LAMBDA(p, ISNUMBER(MATCH(p, bus.allowed, 0)))))), "
    + "SUM(MAP(bus.pins, LAMBDA(p, IF(p >= 2, 1, 0)))) <= ROWS(bus.pins)), "
    + "INDEX(bus.pins, 0 + 1) + SUM(bus.pins) < MAX(bus.pins) * MIN(bus.pins) + 100)");
  assert(F.same(term(XT.backward(there)), collections));
  assert(XText.ToText(term(XT.forward(unique))) === "=ROWS(UNIQUE(bus.pins)) = ROWS(bus.pins)"); // an idiom, which does not read back
  for (const b of buses) {
    for (const expression of [collections, unique]) {
      assert(XV.OfAny(term(XT.forward(expression)), { bus: b }) === V.OfAny(expression, { bus: b }), String(b.pins.get()));
    }
  }
  // A list is `VSTACK(a, b)`, a column of its elements, evaluating alike; a mapping has no counterpart.
  const listed = term(E.list_(bus.pins.count(), 2n).sum());
  assert(XText.ToText(term(XT.forward(listed))) === "=SUM(VSTACK(ROWS(bus.pins), 2))" && F.same(term(XT.backward(term(XT.forward(listed)))), listed));
  assert(buses.every((b) => Number(XV.OfAny(term(XT.forward(listed)), { bus: b })) === Number(V.OfAny(listed, { bus: b }))));
  raises(ValueError, () => XT.forward(term(E.mapping(["a", 1n]))), "Patterns mapping has no Excel counterpart");
  const xs = X.name("xs");
  const refused: [unknown, any, string][] = [
    [bus.entries("ports"), XT, "Patterns call 'entries' has no Excel counterpart"],
    [X.function("INDEX", xs, 1n), XT.inverse(), "Excel call 'INDEX' has no Patterns counterpart"], // only INDEX(xs, i + 1) reads back
    [X.function("AND", xs), XT.inverse(), "Excel call 'AND' has no Patterns counterpart"],
    [XT.forward(unique), XT.inverse(), "Excel call 'UNIQUE' has no Patterns counterpart"],
  ];
  for (const [expression, translator, message] of refused) raises(ValueError, () => translator.forward(term(expression)), message);
}

<a id="TRN-14"></a>
## TRN-14 · Collections translate directly between Python, MATLAB and Excel, as they do through Patterns

In [ ]:
{
  for (const [left, right] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Dialect, F.Dialect][]) {
    const source = into(PATTERNS, left, collections); // TRN-10's
    const there = into(left, right, source);
    assert(F.same(there, into(PATTERNS, right, collections)), `${left.name()} ${right.name()}`);
    assert(F.same(into(right, left, there), source), `${left.name()} ${right.name()}`);
  }
  raises(ValueError, () => Translators.between(PYTHON, MATLAB).forward(into(PATTERNS, PYTHON, unique)),
    "Python3 call has no MATLAB counterpart"); // unique's idiom does not translate
  const listed = term(E.list_(1n, E.list_(2n, 3n).count()).count()); // a list: a list display, a concatenation, VSTACK
  for (const [left, right] of [[PYTHON, MATLAB], [PYTHON, EXCEL], [MATLAB, EXCEL]] as [F.Dialect, F.Dialect][]) {
    const source = into(PATTERNS, left, listed);
    assert(F.same(into(left, right, source), into(PATTERNS, right, listed)) && F.same(into(right, left, into(left, right, source)), source));
  }
  raises(ValueError, () => Translators.between(PYTHON, EXCEL).forward(into(PATTERNS, PYTHON, term(E.mapping(["a", 1n])))),
    "Python3 dict has no Excel counterpart"); // a dict display: Python's alone
}